# Document Q&A Agent

A simple, step-by-step notebook for understanding how the agent validates a document, builds an index, answers a question, and shows source metadata.

Run the cells from top to bottom. The indexing and question cells use OpenAI and require `OPENAI_API_KEY` in `.env`.

## 1. Import the agent functions

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Import LlamaIndex directly for exploration.
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.llms.openai import OpenAI

agent_folder = Path.cwd().parent
load_dotenv(agent_folder / ".env")
SUPPORTED_EXTENSIONS = {".csv", ".docx", ".html", ".json", ".md", ".pdf", ".txt"}
print("Notebook dependencies imported.")

Agent functions imported.


## 2. Choose and validate a document

This notebook uses the included PDF sample. Replace the path with any supported document.

In [ ]:
def validate_document(path: Path) -> Path:
    """Confirm that the document exists and uses a supported extension."""

    if not path.is_file():
        raise FileNotFoundError(f"Document not found: {path}")
    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported extension: {path.suffix}")
    return path


document_path = validate_document(
    agent_folder / "documents" / "samples" / "sample_document.pdf"
)
print(f"Document: {document_path.name}")

Document: sample_document.pdf
Extension: .pdf
Exists: True


## 3. Build the document index

LlamaIndex loads the document and creates searchable vector embeddings.

In [ ]:
reader = SimpleDirectoryReader(input_files=[str(document_path)])
documents = reader.load_data()
index = VectorStoreIndex.from_documents(documents)
print(f"Indexed document chunks: {len(documents)}")

📄 Loading and indexing d:\Project\AI_Agent_Agentic_AI_Project\03_Document_QA_Agent\documents\samples\sample_document.pdf...


2026-09-17 21:35:18,022 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✅ Indexed 1 document chunk(s)
Index is ready.


## 4. Ask one question

The query engine retrieves relevant chunks and sends them to the language model.

In [ ]:
question = "What is the workflow?"
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query(question)
print(response.response)
source_nodes = response.source_nodes

2026-09-17 21:35:19,434 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-17 21:35:22,754 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The workflow involves validating the document, loading content with LlamaIndex, creating a vector index, and answering with OpenAI.


## 5. Inspect source metadata

Inspect the filename, title, page, and retrieved chunk text used for the answer.

In [ ]:
for chunk_number, source in enumerate(source_nodes, start=1):
    metadata = source.node.metadata or {}
    print(f"Chunk: {chunk_number}")
    print(f"Document: {metadata.get('file_name', document_path.name)}")
    print(f"Title: {metadata.get('title', document_path.name)}")
    print(f"Page: {metadata.get('page_label', 'not available')}")
    print(f"Text: {source.node.get_content()[:300]}")
    print("-" * 60)

Chunk: 1
Document: sample_document.pdf
Title: sample_document.pdf
Page: 1
Text: Document Q&A Agent Sample
The Document Q&A Agent answers questions about local documents.
Supported formats: PDF, DOCX, TXT, Markdown, CSV, JSON, and HTML.
Workflow: Validate the document, load content with LlamaIndex, create a vector index, and answer with OpenAI.
------------------------------------------------------------


## 6. Optional follow-up chat

The chat engine keeps memory for follow-up questions.

In [ ]:
chat_engine = index.as_chat_engine(
    chat_mode="context",
    llm=OpenAI(model="gpt-4o-mini", temperature=0),
    memory=ChatMemoryBuffer.from_defaults(token_limit=4096),
)
follow_up = chat_engine.chat("Explain that workflow in one sentence.")
print(follow_up.response)

2026-09-17 21:35:30,270 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-17 21:35:32,212 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The workflow involves validating the document, loading its content using LlamaIndex, creating a vector index, and then answering questions using OpenAI.
